# Data Overview

<a href="https://colab.research.google.com/github/ipavlopoulos/diachronic-greek-letterforms/blob/main/data/load.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook checks the released data folders and shows small metadata/file samples. GitHub's notebook preview is static; use Colab or local Jupyter to run the cells.

In [ ]:
import os
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/ipavlopoulos/diachronic-greek-letterforms.git"
REPO_DIR = Path("/content/diachronic-greek-letterforms")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

if Path("data/palitchar").exists():
    DATA_DIR = Path("data")
elif Path("palitchar").exists():
    DATA_DIR = Path(".")
else:
    raise FileNotFoundError("Could not find the released data directory.")

DATA_DIR.resolve()

## Released CSV Metadata

Hell-Char and PaLit-Char include CSV metadata files. Med-Char is released as cliplets; this notebook derives a lightweight manifest from filenames.

In [ ]:
for dataset in ["hellchar", "palitchar"]:
    csv_path = DATA_DIR / dataset / f"{dataset}.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        print(f"{dataset}: {df.shape[0]} rows, {df.shape[1]} columns")
        display(df.head())
    else:
        print(f"{dataset}: metadata CSV not found at {csv_path}")

## Cliplet Counts

Each cliplet filename begins with the letter label. Med-Char filenames also encode century-like manuscript identifiers such as `09003`, `12039`, and `14007`; the first two digits are used below as a compact century grouping.

In [ ]:
def cliplet_manifest(dataset):
    cliplet_dir = DATA_DIR / dataset / "cliplets"
    rows = []
    for path in sorted(cliplet_dir.glob("*")):
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        parts = path.stem.split("_")
        rows.append({
            "dataset": dataset,
            "filename": path.name,
            "letter": parts[0],
            "document_id": parts[1] if len(parts) > 1 else None,
            "century_code": parts[1][:2] if dataset == "medchar" and len(parts) > 1 else None,
        })
    return pd.DataFrame(rows)

manifests = {name: cliplet_manifest(name) for name in ["hellchar", "palitchar", "medchar"]}
for name, manifest in manifests.items():
    print(f"{name}: {len(manifest)} cliplets")
    display(manifest.head())

In [ ]:
medchar_manifest = manifests["medchar"]
medchar_counts = (
    medchar_manifest
    .groupby(["century_code", "letter"])
    .size()
    .rename("n")
    .reset_index()
    .sort_values(["century_code", "letter"])
)

print("Med-Char counts by century code and letter:")
display(medchar_counts.head(30))